In [1]:
# imports :
import pandas as pd
import re
import nltk

In [2]:
# télécharger ressources :
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\berna\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\berna\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
# charger dataset :
dataset = pd.read_csv("../data/dataset_final.csv")

dataset.head()

,text,label
0,Message-ID: <18782981.1075855378110.JavaMail.e...,ham
1,Message-ID: <15464986.1075855378456.JavaMail.e...,ham
2,Message-ID: <24216240.1075855687451.JavaMail.e...,ham
3,Message-ID: <13505866.1075863688222.JavaMail.e...,ham
4,Message-ID: <30922949.1075863688243.JavaMail.e...,ham


In [4]:
# outils nlp :
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()

In [5]:
# Fonction de nettoyage :
def clean_text(text):
    
    text = text.lower()
    
    # supprimer HTML
    text = re.sub(r"<.*?>", " ", text)
    
    # supprimer URLs
    text = re.sub(r"http\S+|www\S+", " ", text)
    
    # supprimer emails
    text = re.sub(r"\S+@\S+", " ", text)
    
    # supprimer chiffres
    text = re.sub(r"\d+", " ", text)
    
    # supprimer ponctuation
    text = re.sub(r"[^\w\s]", " ", text)
    
    # supprimer espaces multiples
    text = re.sub(r"\s+", " ", text).strip()
    
    # tokenisation
    words = text.split()
    
    # stopwords + lemmatisation
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]
    
    return " ".join(words)

In [6]:
# test rapide :
sample = dataset["text"].iloc[0]

print("AVANT :")
print(sample)

print("\nAPRÈS :")
print(clean_text(sample))

AVANT :
Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>
Date: Mon, 14 May 2001 16:39:00 -0700 (PDT)
From: phillip.allen@enron.com
To: tim.belden@enron.com
Subject: 
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: Tim Belden <Tim Belden/Enron@EnronXGate>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).pst

Here is our forecast

 

APRÈS :
message id date mon may pdt subject mime version content type text plain charset u ascii content transfer encoding bit x phillip k allen x tim belden x cc x bcc x folder phillip_allen_jan _ allen phillip k sent mail x origin allen p x filename pallen non privileged pst forecast


In [7]:
# appliquer tout sur le dataset :
dataset["clean_text"] = dataset["text"].apply(clean_text)

In [8]:
# vérification :
dataset[["text","clean_text"]].sample(5)

,text,clean_text
204143,axel thimm axelthimmphysikfuberlinde mon oct 0...,axel thimm axelthimmphysikfuberlinde mon oct p...
84373,"So then, Neale Pickett is all like: > Maybe t...",neale pickett like maybe subtle interaction ge...
43314,Message-ID: <2484623.1075857864893.JavaMail.ev...,message id date thu nov pst subject weekend ou...
137195,gone night got check new site somebody taught ...,gone night got check new site somebody taught ...
37239,Message-ID: <17237580.1075840376293.JavaMail.e...,message id date wed feb pst houston trading su...


In [9]:
# save :
dataset.to_csv("../data/dataset_clean.csv", index=False)

In [10]:
IMPORTANT_WORDS = {
    "bank", "account", "verify", "password", "login",
    "urgent", "click", "security", "update", "confirm"
}

In [11]:
def clean_text_advanced(text):
    
    text = text.lower()
    
    # HTML
    text = re.sub(r"<.*?>", " ", text)
    
    # URLs (on remplace par token)
    text = re.sub(r"http\S+|www\S+", " URL ", text)
    
    # emails
    text = re.sub(r"\S+@\S+", " EMAIL ", text)
    
    # chiffres
    text = re.sub(r"\d+", " ", text)
    
    # ponctuation
    text = re.sub(r"[^\w\s]", " ", text)
    
    # espaces
    text = re.sub(r"\s+", " ", text).strip()
    
    words = text.split()
    
    clean_words = []
    
    for word in words:
        
        # garder si important
        if word in IMPORTANT_WORDS:
            clean_words.append(word)
            continue
        
        # supprimer stopwords sauf importants
        if word not in stop_words:
            clean_words.append(lemmatizer.lemmatize(word))
    
    return " ".join(clean_words)

In [12]:
dataset["clean_text"] = dataset["text"].apply(clean_text_advanced)

In [13]:
dataset["has_url"] = dataset["text"].apply(lambda x: 1 if "http" in x else 0)

dataset["has_email"] = dataset["text"].apply(lambda x: 1 if "@" in x else 0)

dataset["text_length"] = dataset["text"].apply(len)

dataset["num_words"] = dataset["text"].apply(lambda x: len(str(x).split()))

In [14]:
SUSPICIOUS_WORDS = [
    "urgent", "verify", "password", "bank",
    "account", "click", "login", "confirm"
]

for word in SUSPICIOUS_WORDS:
    dataset[f"has_{word}"] = dataset["clean_text"].apply(
        lambda x: 1 if word in x else 0
    )

In [15]:
dataset.sample(5)

,text,label,clean_text,has_url,has_email,text_length,num_words,has_urgent,has_verify,has_password,has_bank,has_account,has_click,has_login,has_confirm
209444,faith zi7wynx2g1686hotmailcom save 75 term lif...,phishing,faith zi wynx g hotmailcom save term life insu...,1,0,584,82,0,0,0,0,0,1,0,0
201215,jesse noller kirgqrbgmailcom sun may 4 2008 95...,phishing,jesse noller kirgqrbgmailcom sun may wrote sni...,1,0,1146,141,0,0,0,0,0,0,0,0
175108,stephen williams wvilignet finally tacoma narr...,phishing,stephen williams wvilignet finally tacoma narr...,1,0,5196,593,0,0,0,0,0,0,0,0
86470,bugzilla-daemon@hughes-family.org wrote: > htt...,spam,EMAIL wrote URL ok fixed cheer fix creating pi...,1,1,403,56,0,0,0,0,0,0,0,0
41270,Message-ID: <10347561.1075851887711.JavaMail.e...,ham,message id date mon may pdt EMAIL EMAIL EMAIL ...,0,1,1122,88,0,0,0,0,0,0,0,0


In [17]:
dataset.to_csv("../data/dataset_clean_advanced.csv", index=False)